# Balloon/Satellite Detection of HNL Decays

This notebook calculates the sensitivity of a balloon/satellite detector to Heavy Neutral Lepton (HNL) decays.

**Setup:**
- Muon beam travels upward through Earth toward a satellite/balloon detector
- HNLs are produced when muons scatter off nucleons in a target region below the surface
- HNLs travel upward and decay in the atmosphere
- Cherenkov light from decay products (muons) is detected by the satellite

**Sections:**
1. Setup and imports
2. Cherenkov photon detection verification
3. HNL flux geometry and signal calculation
4. Sensitivity estimation

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import os

plt.style.use("figures.mplstyle")

from src.constants import *
from src.xs_and_decays import *
from src.balloon import *
from src.cherenkov import (
    cherenkov_photons_detected_vectorized,
    get_cherenkov_angle,
    get_cherenkov_yield_per_meter,
    CherenkovLookupTable
)

# Beam parameters
E_mu = 5000  # GeV - muon beam energy
E_N = E_mu / 2  # Approximate HNL energy

print(f"Muon beam energy: {E_mu} GeV")
print(f"Detector altitude: {L_det/1000:.0f} km")
print(f"Detector radius: {R_det} m")
print(f"Cherenkov angle: {np.degrees(theta_C):.4f} deg")
print(f"Cherenkov yield: {Ch_dN_dx:.2f} photons/m")

## 2. Cherenkov Photon Detection

Verify the Cherenkov photon calculation for different particle configurations.

In [ ]:
# Create a 2D map: photon count as function of (z, theta)
z_range_map = np.logspace(2, np.log10(L_det), 20)  # 100 m to detector altitude
theta_range_map = np.linspace(0, 0.05, 20)  # 0 to ~3 degrees
track_length = 2*L_det  # m

for x0 in [0,100,500]:

    N_photons_map = np.zeros((len(z_range_map), len(theta_range_map)))

    print("Computing photon map...")
    for iz, z in enumerate(z_range_map):
        for ith, theta in enumerate(theta_range_map):
            r_0 = np.array([-x0, 0, -z])
            p_hat = np.array([np.sin(theta), 0, np.cos(theta)])
            N_photons_map[iz, ith], _, _ = cherenkov_photons_detected_vectorized(
                r_0, p_hat, track_length, R_det, N_psi=2000, N_track=5000
            )
        if (iz + 1) % 10 == 0:
            print(f"  {iz+1}/{len(z_range_map)} complete")

    print("Done!")

    # Plot
    theta_mesh, z_mesh = np.meshgrid(np.degrees(theta_range_map), z_range_map/1000)

    plt.figure(figsize=(10, 6))
    pcm = plt.pcolormesh(theta_mesh, z_mesh, N_photons_map,
                        norm=LogNorm(vmin=1, vmax=N_photons_map.max()),
                        shading='auto', cmap='viridis')
    # plt.axvline(np.degrees(theta_C), color='r', linestyle='--', linewidth=2,
    #             label=f'Cherenkov angle = {np.degrees(theta_C):.2f} deg')
    plt.colorbar(pcm, label='Cherenkov photons detected')
    plt.xlabel('Particle angle from vertical (degrees)')
    plt.ylabel('Distance below detector (km)')
    plt.yscale('log')
    plt.text(0,z_range_map[0]/1e3,
            "Initial Transverse Displacement: %1.1f m"%x0 + "\n" + \
            r"$\theta_{\rm Ch}^{\rm air} = %1.1f$ deg"%np.degrees(theta_C) + "\n" + \
            r"$R_{\rm det} = %1.1f$ m"%R_det,
            fontsize=14)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig("Figures/balloon/cherenkov_photon_map_x0_%1.1f_m.png"%x0, dpi=300)
    plt.show()



## 3. HNL Flux Geometry and Signal Calculation

Set up the full HNL production and detection geometry, accounting for:
- Angular spread from HNL production
- HNL decay kinematics
- Realistic muon track lengths in atmosphere

In [ ]:
# Create default flux geometry
flux_geometry = HNLFluxGeometry(
    E_mu=E_mu,
    dump_angle=1.53, # radians
    satellite_height=100000  # 10 km altitude
)

print("HNL Flux Geometry Configuration")
print("=" * 40)
print(f"Muon beam energy: {flux_geometry.E_mu} GeV")
print(f"Beam direction: {flux_geometry.beam_dir}")
print(f"Target depth: {flux_geometry.dump_depth} m")
print(f"Target length: {flux_geometry.L_target} m")
print(f"Satellite height: {flux_geometry.satellite_height/1000:.1f} km")

In [ ]:
# Test signal calculation for a single HNL configuration
m_N_test = 20  # GeV
U2_test = 5e-12
N_samples_test = 1000

print(f"Testing signal calculation for m_N = {m_N_test} GeV, U2 = {U2_test:.0e}")
print("-" * 50)

results = compute_signal_at_satellite(
    m_N_test, E_mu, U2_test, flux_geometry, N_samples=N_samples_test
)

for min_ph in [1, 5, 10, 50]:
    det_eff, mean_ph, n_events = summarize_signal(
        results[0], results[2], N_samples_test, min_ph, results[3], results[1]
    )
    print(f"  min_photons={min_ph:3d}: eff={det_eff:.4f}, <N_ph>={mean_ph:.1f}, events={n_events:.2f}")

## 4. Sensitivity Estimation

Scan over HNL mass and mixing angle to estimate detector sensitivity.

In [ ]:
def compute_sensitivity_scan(flux_geometry, m_N_range, U2_range,
                              N_samples=1000, verbose=True):
    """
    Scan over HNL mass and mixing angle, storing raw photon counts.

    The expensive Cherenkov simulation is run once per (m_N, U2) point.
    Use apply_threshold_to_scan() afterwards to get events/efficiency
    for any photon threshold without re-running.
    """
    n_m = len(m_N_range)
    n_U2 = len(U2_range)

    # Store raw MC output per grid point
    photon_counts_grid = [[None]*n_m for _ in range(n_U2)]
    N_HNLs_per_muon_grid = np.zeros((n_U2, n_m))
    weights_grid = [[None]*n_m for _ in range(n_U2)]

    for im, m_N in enumerate(m_N_range):
        if verbose:
            print(f"Scanning m_N = {m_N:.2f} GeV ({im+1}/{n_m})")

        for iU2, U2 in enumerate(U2_range):
            try:
                ph_counts, decay_weights, n_hnl, ch_weight, decay_points = compute_signal_at_satellite(
                    m_N, flux_geometry.E_mu, U2, flux_geometry,
                    N_samples=N_samples
                )
                photon_counts_grid[iU2][im] = ph_counts
                N_HNLs_per_muon_grid[iU2, im] = n_hnl
                weights_grid[iU2][im] = decay_weights * ch_weight
            except Exception as e:
                if verbose:
                    print(f"  Warning: Failed for U2={U2:.0e}: {e}")
                photon_counts_grid[iU2][im] = np.zeros(N_samples)
                weights_grid[iU2][im] = np.zeros(N_samples)
                N_HNLs_per_muon_grid[iU2, im] = 0.0

    return {
        'm_N': m_N_range,
        'U2': U2_range,
        'N_samples': N_samples,
        'photon_counts': photon_counts_grid,
        'N_HNLs_per_muon': N_HNLs_per_muon_grid,
        'weights': weights_grid,
    }


def apply_threshold_to_scan(results, min_photons=10):
    """
    Apply a photon threshold to pre-computed scan results.

    This is fast -- no Cherenkov re-simulation needed.
    Returns a new dict with 'events', 'efficiency', 'mean_photons' arrays.
    """
    m_N_range = results['m_N']
    U2_range = results['U2']
    N_samples = results['N_samples']
    ph_grid = results['photon_counts']
    n_hnl_grid = results['N_HNLs_per_muon']
    weights_grid = results['weights']

    n_U2, n_m = len(U2_range), len(m_N_range)
    events = np.zeros((n_U2, n_m))
    efficiency = np.zeros((n_U2, n_m))
    mean_photons = np.zeros((n_U2, n_m))

    for iU2 in range(n_U2):
        for im in range(n_m):
            eff, mph, n_ev = summarize_signal(
                ph_grid[iU2][im], n_hnl_grid[iU2, im],
                N_samples, min_photons=min_photons,
                decay_weights=weights_grid[iU2][im]
            )
            events[iU2, im] = n_ev
            efficiency[iU2, im] = eff
            mean_photons[iU2, im] = mph

    return {
        'm_N': m_N_range,
        'U2': U2_range,
        'events': events,
        'efficiency': efficiency,
        'mean_photons': mean_photons,
    }


def find_sensitivity_contour(results, n_events_threshold=3):
    """
    Find the sensitivity contour (U2 vs m_N) for a given event threshold.

    Parameters
    ----------
    results : dict
        Output from apply_threshold_to_scan (must contain 'events')
    n_events_threshold : float
        Number of events for sensitivity (default 3 for 95% CL)

    Returns
    -------
    m_N_contour, U2_contour : arrays
        Mass and mixing angle values along the contour
    """
    m_N_range = results['m_N']
    U2_range = results['U2']
    events = results['events']

    m_N_contour = []
    U2_contour = []

    for im, m_N in enumerate(m_N_range):
        # Find U2 where events crosses threshold
        ev_col = events[:, im]

        # Find first U2 where we get enough events
        above_threshold = ev_col >= n_events_threshold
        if np.any(above_threshold):
            idx = np.where(above_threshold)[0][-1]  # Largest U2 below threshold
            if idx < len(U2_range) - 1:
                # Interpolate
                U2_sens = np.interp(n_events_threshold,
                                    [ev_col[idx+1], ev_col[idx]],
                                    [U2_range[idx+1], U2_range[idx]])
            else:
                U2_sens = U2_range[idx]
            m_N_contour.append(m_N)
            U2_contour.append(U2_sens)

    return np.array(m_N_contour), np.array(U2_contour)


def plot_results(results,
                 plot_keys = ['events','efficiency','mean_photons'],
                 title=None,
                 save_path=None):
    """
    Plot sensitivity contours from scan results.
    """

    m_N_range = results['m_N']
    U2_range = results['U2']

    for key in plot_keys:

        plottable = results[key]
        if key=="events": levels = [1, 10, 100, 1000]
        elif key=="efficiency": levels = [1e-3,1e-2,1e-1,1]
        elif key=="mean_photons": levels = [50,100,500,1000]
        else:
            print("Invalid plot key %s, returning..."%key)
            return

        m_N_mesh, U2_mesh = np.meshgrid(m_N_range, U2_range)

        fig, ax = plt.subplots(figsize=(10, 7))

        if key in ["mean_photons","efficiency"]:
            plt.pcolormesh(m_N_mesh,U2_mesh,plottable,cmap="Purples",norm=LogNorm())
            c = plt.colorbar()
            c.set_label(key)


        if key in ["events"]:

            # Contour levels
            colors = ['red', 'orange', 'green', 'blue']

            for level, color in zip(levels, colors):
                cs = ax.contour(m_N_mesh, U2_mesh, plottable, levels=[level],
                                colors=[color], linewidths=2)
                ax.clabel(cs, fmt=f'{level:.0f} events', fontsize=10)

        # Seesaw bound
        U2_seesaw = 5e-11 / m_N_range
        ax.fill_between(m_N_range, 1e-20, U2_seesaw, color='gray', alpha=0.2)
        ax.plot(m_N_range, U2_seesaw, 'gray', linestyle='--', label='Seesaw bound')

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'HNL Mass $m_N$ [GeV]')
        ax.set_ylabel(r'$|U_\mu|^2$')
        ax.set_xlim(m_N_range[0], m_N_range[-1])
        ax.set_ylim(U2_range[0], U2_range[-1])
        ax.legend(loc='lower left')

        if title:
            ax.set_title(title)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path+"_%s.png"%key, dpi=300)

        plt.show()
        plt.close(fig)

In [ ]:
# Run sensitivity scan (expensive -- stores raw photon counts)
m_N_scan = np.array([5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 60, 70, 80, 90])
U2_scan = np.logspace(-14, -7, 5)  # Wide range of mixing angles

print("Running sensitivity scan...")
print(f"Mass range: {m_N_scan[0]:.1f} - {m_N_scan[-1]:.1f} GeV")
print(f"U2 range: {U2_scan[-1]:.0e} - {U2_scan[0]:.0e}")
print()

raw_results = compute_sensitivity_scan(
    flux_geometry, m_N_scan, U2_scan,
    N_samples=1000, verbose=True
)

print("\nScan complete!")

In [ ]:
# Apply threshold and plot -- change min_photons here to explore different cuts
results = apply_threshold_to_scan(raw_results, min_photons=10)

plot_results(
    results,
    save_path='Figures/balloon/sensitivity_scan'
)

## Simulated HNL Kinematics

In [ ]:
mc = {}
for mass in [5,6,7,8,9,10,12,14,16,20,25,30,40,50,60,70,80,90]:
    filename = "data/HNL_kinematics/Momentum%2.1f.dat" % mass
    mc[mass] = pd.read_csv(filename,sep='\s+')

In [ ]:
cmap = cm.get_cmap('plasma', len(mc))
for i, (mass, df) in enumerate(mc.items()):
    P_tot = np.sqrt(df.PNx**2 + df.PNy**2 + df.PNz**2)
    plt.hist(np.arccos(df.PNz/P_tot), bins=np.logspace(-6, -1, 100), label=f'm_N={mass} GeV', histtype='step', color=cmap(i))
plt.loglog()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Angle from beam direction (radians)')
plt.ylabel('Number of events')
plt.show()

for i, (mass, df) in enumerate(mc.items()):
   plt.hist(df.PNe, bins=np.logspace(1, 4, 100), label=f'm_N={mass} GeV', histtype='step', color=cmap(i))
plt.loglog()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('HNL Energy (GeV)')
plt.ylabel('Number of events')
plt.show()

# Cherenkov transmission studies

In [ ]:
from src.cherenkov import cherenkov_transmission

In [ ]:
altitude_range = np.linspace(0, 30000, 150)
zenith_range = np.linspace(0, np.radians(80), 100)
transmission_map = np.zeros((len(altitude_range), len(zenith_range)))
for i, alt in enumerate(altitude_range):
    for j, zen in enumerate(zenith_range):
        transmission_map[i, j] = cherenkov_transmission(alt, zen)

plt.pcolormesh(altitude_range/1000, np.degrees(zenith_range), transmission_map.T, shading='auto', cmap='plasma')
c = plt.colorbar()
plt.xlabel('Altitude (km)')
plt.ylabel('Zenith angle (degrees)')
c.set_label('Cherenkov transmission')
plt.show()

# Background studies and optimization of selection criteria for signal extraction.

In [ ]:
from src.background import compute_background_at_satellite

detector_positions = np.array([[0,0,20e3],
                               [0,0,100e3],
                               #[0,500,150e3],
                               #[0,-500,100e3],
                               #[0,0,100e3]
                               ]) # km

geom = HNLFluxGeometry(
        E_mu=E_mu,
        dump_depth=100,
        dump_angle = 1.53
        )

In [ ]:

sample_mN = 20  # GeV
sample_U2 = 1e-11



N_samples = 10000

class experiment_results:

    def __init__(self, photon_counts, interaction_weights, N_per_muon,
                 cherenkov_weight, interaction_positions):
        self.photon_counts = photon_counts
        self.interaction_weights = interaction_weights
        self.N_per_muon = N_per_muon
        self.cherenkov_weight = cherenkov_weight
        self.interaction_positions = interaction_positions
        self.weights = interaction_weights * N_per_muon * cherenkov_weight * N_muon_decays / N_samples

background_results = experiment_results(*compute_background_at_satellite(geom,
                                                          N_samples=N_samples,
                                                          detector_positions=detector_positions,
                                                          uniform_gen=True
                                                         )
)

signal_results = experiment_results(*compute_signal_at_satellite(sample_mN, E_mu, sample_U2, geom,
                                                              N_samples=N_samples,
                                                              detector_positions=detector_positions,
                                                              uniform_gen=True)
)

In [ ]:
for ib,h in enumerate(detector_positions[:,-1]):

    plt.plot([], [], color='green', label='Background', linestyle='-')
    plt.plot([], [], color='red', label=r'Signal ($m_N = %d$ GeV, $U^2 = %1.0e$)'%(sample_mN,sample_U2), linestyle='-')
    transverse_bins = np.linspace(0,5000,50)
    for min_photons,ls in [(10, '-'),(50, ':')]:
        plt.plot([], [], color='black', linestyle=ls, label='> %d photons'%min_photons)
        detected = background_results.photon_counts[ib] > min_photons
        n_bkg,_,_ = plt.hist(np.sqrt(background_results.interaction_positions[:,0]**2 + background_results.interaction_positions[:,1]**2),
                             bins=transverse_bins,weights=background_results.weights*detected,color='green',histtype='step',linestyle=ls)
        sig_detected = signal_results.photon_counts[ib] > min_photons
        n_sig,_,_ = plt.hist(np.sqrt(signal_results.interaction_positions[:,0]**2 + signal_results.interaction_positions[:,1]**2),
                             bins=transverse_bins,weights=signal_results.weights*sig_detected,color='red',histtype='step',linestyle=ls,label=None)
        plt.step(transverse_bins[:-1], n_sig+n_bkg, where='post', color='black', linestyle=ls)
    plt.semilogy()
    plt.xlabel("transverse pos [m]")
    plt.ylabel("Events / 1e22 muons")
    plt.text(0.55, 0.95, f"Detector z pos: {h/1000:.1f} km", transform=plt.gca().transAxes, fontsize=14)
    plt.legend(bbox_to_anchor=(0.1, 1.4), loc='upper left')
    plt.show()


    plt.plot([], [], color='green', label='Background', linestyle='-')
    plt.plot([], [], color='red', label=r'Signal ($m_N = %d$ GeV, $U^2 = %1.0e$)'%(sample_mN,sample_U2), linestyle='-')
    hbins = np.linspace(0,h,50)
    for min_photons,ls in [(10, '-'),(50, ':')]:
        plt.plot([], [], color='black', linestyle=ls, label='> %d photons'%min_photons)
        bkg_detected = background_results.photon_counts[ib] > min_photons
        n_bkg,_,_ = plt.hist(background_results.interaction_positions[:,-1],bins=hbins,weights=background_results.weights*bkg_detected,color='green',histtype='step',linestyle=ls)
        sig_detected = signal_results.photon_counts[ib] > min_photons
        n_sig,_,_ = plt.hist(signal_results.interaction_positions[:,-1],bins=hbins,weights=signal_results.weights*sig_detected,color='red',histtype='step',linestyle=ls,label=None)
        plt.step(hbins[:-1], n_sig+n_bkg, where='post', color='black', linestyle=ls)
    plt.semilogy()
    plt.xlabel("longitudinal pos [m]")
    plt.ylabel("Events / 1e22 muons")
    plt.text(0.55, 0.95, f"Detector z pos: {h/1000:.1f} km", transform=plt.gca().transAxes, fontsize=14)
    plt.legend(bbox_to_anchor=(0.1, 1.4), loc='upper left')
    plt.show()

In [ ]:
plt.hist2d(background_results.photon_counts[0],background_results.photon_counts[1],
           bins=np.linspace(0,200,200),norm=LogNorm(),weights=background_results.weights)
plt.show()

plt.hist2d(signal_results.photon_counts[0],signal_results.photon_counts[1],
           bins=np.linspace(0,200,200),norm=LogNorm(),weights=signal_results.weights)
plt.show()

In [ ]:

plt.plot([], [], color='green', label='Background', linestyle='-')
plt.plot([], [], color='red', label=r'Signal ($m_N = %d$ GeV, $U^2 = %1.0e$)'%(sample_mN,sample_U2), linestyle='-')
for min_photons,ls in [(10, '-'), (100, ':')]:
    plt.plot([], [], color='black', linestyle=ls, label='> %d photons'%min_photons)
    detected = np.logical_and(background_results.photon_counts[0] == 0,
                              background_results.photon_counts[1] > min_photons)
    plt.hist(background_results.interaction_positions[:,-1],bins=np.linspace(0,h,50),weights=background_results.weights*detected,color='green',histtype='step',linestyle=ls)
    sig_detected = signal_results.photon_counts[ib] > min_photons
    plt.hist(signal_results.interaction_positions[:,-1],bins=np.linspace(0,h,50),weights=signal_results.weights*sig_detected,color='red',histtype='step',linestyle=ls,label=None)
plt.semilogy()
plt.xlabel("Altitude [m]")
plt.ylabel("Events / 1e22 muons")
plt.text(0.55, 0.95, f"Detector altitude: {h/1000:.1f} km", transform=plt.gca().transAxes, fontsize=14)
plt.legend(bbox_to_anchor=(0.1, 1.4), loc='upper left')
plt.show()

plt.plot([], [], color='green', label='Background', linestyle='-')
plt.plot([], [], color='red', label=r'Signal ($m_N = %d$ GeV, $U^2 = %1.0e$)'%(sample_mN,sample_U2), linestyle='-')
plt.plot([], [], color='black', label='Detector 1', linestyle=':')
plt.plot([], [], color='black', label='Detector 2', linestyle='--')
plt.hist(background_results.photon_counts[0],bins=np.linspace(0,1000,50),weights=background_results.weights,color='green',histtype='step',linestyle=":")
plt.hist(background_results.photon_counts[1],bins=np.linspace(0,1000,50),weights=background_results.weights,color='green',histtype='step',linestyle="--")
plt.hist(signal_results.photon_counts[0],bins=np.linspace(0,1000,50),weights=signal_results.weights,color='red',histtype='step',linestyle=":")
plt.hist(signal_results.photon_counts[1],bins=np.linspace(0,1000,50),weights=signal_results.weights,color='red',histtype='step',linestyle="--")
plt.semilogy()
plt.xlabel("Number of Cherenkov photons detected")
plt.ylabel("Events / 1e22 muons")
plt.text(0.55, 0.95, f"Detector altitude: {h/1000:.1f} km", transform=plt.gca().transAxes, fontsize=14)
plt.legend(bbox_to_anchor=(0.1, 1.4), loc='upper left')
plt.show()

# beam dump geometry 

In [ ]:
# some geometry

R_earth = 6371e3  # m

def dump_length(dump_depth, dump_angle):
    a = 1
    b = -2*(R_earth - dump_depth)*np.cos(np.pi - dump_angle)
    c = (R_earth-dump_depth)**2 - R_earth**2
    return (-b + np.sqrt(b**2 - 4*a*c)) / (2*a)

def decay_length(dump_depth, dump_angle, balloon_height=5e3):
    dump_length_val = dump_length(dump_depth, dump_angle)
    a = 1
    b = 2*(dump_length_val - np.cos(np.pi - dump_angle) * (R_earth - dump_depth))
    c = (R_earth - dump_depth)**2 + dump_length_val**2 - 2*dump_length_val*(R_earth - dump_depth)*np.cos(np.pi - dump_angle) - (R_earth + balloon_height)**2
    return (-b + np.sqrt(b**2 - 4*a*c)) / (2*a)

In [ ]:
dump_distance_range = np.linspace(1,200,100)
dump_angle_range = np.linspace(0,1,100)
dump_length_map = np.zeros((len(dump_distance_range), len(dump_angle_range)))
decay_length_map = np.zeros((len(dump_distance_range), len(dump_angle_range)))
for i, d in enumerate(dump_distance_range):
    for j, a in enumerate(dump_angle_range):

        dump_length_map[i,j] = dump_length(d, np.pi/2-a)
        decay_length_map[i,j] = decay_length(d, np.pi/2-a)

In [ ]:
plt.pcolormesh(dump_distance_range, dump_angle_range, dump_length_map.T, shading='auto', cmap='viridis', norm=LogNorm())
plt.colorbar(label='Dump length (m)')
plt.contour(dump_distance_range, dump_angle_range, dump_length_map.T, levels=[1000,10000], colors='black', linestyles=['-', '--'])
plt.xlabel("Dump depth (m)")
plt.ylabel("Dump angle (radians)")
plt.show()

plt.pcolormesh(dump_distance_range, dump_angle_range, decay_length_map.T, shading='auto', cmap='viridis', norm=LogNorm())
plt.colorbar(label='Decay length (m)')
plt.contour(dump_distance_range, dump_angle_range, decay_length_map.T, levels=[10000,100000], colors='black', linestyles=['-', '--'])

plt.xlabel("Dump depth (m)")
plt.ylabel("Dump angle (radians)")
plt.show()

In [ ]:
balloon_height_low = 1000  # m
balloon_height_high = 5000  # m
for i,dump_depth in enumerate([10, 50, 100, 200]):
    color = plt.cm.Dark2(i)

    dump_angle_range = np.linspace(-np.pi/2, np.pi/2, 1000)
    dump_length_vals = np.array([dump_length(dump_depth, np.pi/2 - a) for a in dump_angle_range])
    decay_length_vals_low = np.array([decay_length(dump_depth, np.pi/2 - a, balloon_height_low) for a in dump_angle_range])
    decay_length_vals_high = np.array([decay_length(dump_depth, np.pi/2 - a, balloon_height_high) for a in dump_angle_range])
    plt.fill_between(dump_length_vals/1e3, decay_length_vals_low/1e3, decay_length_vals_high/1e3, color=color, alpha=0.5, label=f'Dump depth = {dump_depth} m')
plt.loglog()
plt.xlabel(r'$\mu$Dump length (km)')
plt.ylabel('Decay region length (km)')
plt.legend(bbox_to_anchor=(1.0, 1.3), title=r"Balloon height $\in [%d,%d]$ km" % (balloon_height_low/1e3, balloon_height_high/1e3), ncol=2)
plt.savefig("Figures/balloon/dump_decay_geometry.png", dpi=300, bbox_inches='tight')
plt.show()

## Which background dominates, decay or scattering production of neutrinos?

In [ ]:
def decay_integrand(l,E_mu_init):
    E_mu_current = muon_energy_in_earth(E_mu_init,l*1e-2)
    ldec = 6.2*10**5 * E_mu_current
    return np.exp(-l/ldec)/ldec

def P_dec(L,E_mu_init):
    return quad(decay_integrand,0,L,args=E_mu_init)[0]

def scatter_integrand(l,E_mu_init):
    E_mu_current = muon_energy_in_earth(E_mu_init,l*1e-2)
    return 0.68e-38 * E_mu_current * n_earth

def P_scat(L,E_mu_init):
    return quad(scatter_integrand,0,L,args=E_mu_init)[0]
    #return 0.68e-38*n_earth * (E_mu_init * L - 1e-3 * L**2)

def mcs_angle(L,E_mu_init):
    X0 = 26.5 # g/cm^2 for rock
    x = L * 2.65  # g/cm^2
    return 0.0136 * np.sqrt(x/X0) * (1 + 0.038 * np.log(x/X0)) / E_mu_init

In [ ]:
E_mu_init_range = np.logspace(2,5,50)
L_range = np.logspace(1,3.2,50)*1e2 # cm
P_scat_array = np.zeros((len(E_mu_init_range),len(L_range)))
P_dec_array = np.zeros((len(E_mu_init_range),len(L_range)))
for i,E_mu_init in enumerate(E_mu_init_range):
    for j,L in enumerate(L_range):
        P_scat_array[i,j] = P_scat(L,E_mu_init)
        P_dec_array[i,j] = P_dec(L,E_mu_init)


In [ ]:
plt.pcolor(E_mu_init_range,L_range/1e5,P_dec_array.T,norm=LogNorm())
c = plt.colorbar()
c.set_label("Neutrinos per muon (decay)")
plt.xlabel("Muon energy (GeV)")
plt.ylabel("Path length in Earth (km)")
plt.loglog()
plt.show()
plt.pcolor(E_mu_init_range,L_range/1e5,P_scat_array.T,norm=LogNorm())
c = plt.colorbar()
c.set_label("Neutrinos per muon (scatter)")
plt.xlabel("Muon energy (GeV)")
plt.ylabel("Path length in Earth (km)")
plt.loglog()
plt.show()
plt.pcolor(E_mu_init_range,L_range/1e5,P_dec_array.T/P_scat_array.T,norm=LogNorm(vmin=1e-3,vmax=1e3),cmap='RdBu')
c = plt.colorbar()
c.set_label("Decay/Scatter rate")
plt.xlabel("Muon energy (GeV)")
plt.ylabel("Path length in Earth (km)")
plt.loglog()
plt.show()

In [ ]:
plt.plot(L_range*1e-2,muon_energy_in_earth(5000,L_range*1e-2))
plt.xlabel("Position [m]")
plt.ylabel("Muon energy (GeV)")
plt.show()

plt.plot(L_range*1e-2,mcs_angle(L_range,5000)*1e3)
plt.xlabel("Position [m]")
plt.ylabel("Muon rms angle (mrad)")
plt.show()

In [ ]:
geom = HNLFluxGeometry(
        E_mu=5000,
        dump_depth=100,
        dump_angle = 1.53
        )

In [ ]:
m_N = 0
N_samples = 1000000
samples_decay = geom.sample_production_points_weighted(m_N,1,N_samples,mode='decay')
samples_scatter = geom.sample_production_points_weighted(m_N,1,N_samples,mode='scattering')
muon_energy_decay = muon_energy_in_earth(geom.E_mu, samples_decay[0][:,2]+geom.L_target)
muon_energy_scatter = muon_energy_in_earth(geom.E_mu, samples_scatter[0][:,2]+geom.L_target)

kinematics_decay = geom.sample_kinematics(muon_energy_decay, m_N=None, mode="decay")
kinematics_scatter = geom.sample_kinematics(muon_energy_scatter, m_N=None, mode="scattering")

In [ ]:
Ethresh = 500

plt.hist(kinematics_decay[0],bins=np.linspace(0,5000,100),weights=samples_decay[2]*np.ones_like(samples_decay[0][:,-1])/N_samples,label='Decay',histtype='step',color="dodgerblue")
plt.hist(kinematics_scatter[0],bins=np.linspace(0,5000,100),weights=samples_scatter[2]*np.ones_like(samples_scatter[0][:,-1])/N_samples,label='Scattering',histtype='step',color="orangered")
plt.semilogy()
plt.xlabel("Neutrino energy (GeV)")
plt.ylabel("Neutrinos per muon")
plt.legend()
plt.show()

plt.hist(samples_decay[0][:,-1],bins=np.linspace(-geom.L_target,0,100),weights=samples_decay[2]*np.ones_like(samples_decay[0][:,-1])/N_samples,label='Decay',histtype='step',color="dodgerblue")
plt.hist(samples_scatter[0][:,-1],bins=np.linspace(-geom.L_target,0,100),weights=samples_scatter[2]*np.ones_like(samples_scatter[0][:,-1])/N_samples,label='Scattering',histtype='step',color="orangered")
plt.hist(samples_decay[0][:,-1],bins=np.linspace(-geom.L_target,0,100),weights=samples_decay[2]*np.array(kinematics_decay[0]>Ethresh)/N_samples,histtype='step',color="dodgerblue",ls="--")
plt.hist(samples_scatter[0][:,-1],bins=np.linspace(-geom.L_target,0,100),weights=samples_scatter[2]*np.array(kinematics_scatter[0]>Ethresh)/N_samples,histtype='step',color="orangered",ls="--")
plt.plot([],[],color="black",ls="--",label="E > {} GeV".format(Ethresh))
plt.semilogy()
plt.xlabel("Position along target (m)")
plt.ylabel("Neutrinos per muon")
plt.legend()
plt.show()

plt.hist(np.arccos(kinematics_decay[1][:,-1])*1e3,bins=np.logspace(-4,4,100),weights=samples_decay[2]*np.array(kinematics_decay[0]>0)/N_samples,label='Decay',histtype='step',color="dodgerblue")
plt.hist(np.arccos(kinematics_scatter[1][:,-1])*1e3,bins=np.logspace(-4,4,100),weights=samples_scatter[2]*np.ones(N_samples)/N_samples*np.array(kinematics_scatter[0]>0),label='Scattering',histtype='step',color="orangered")
plt.hist(np.arccos(kinematics_decay[1][:,-1])*1e3,bins=np.logspace(-4,4,100),weights=samples_decay[2]*np.array(kinematics_decay[0]>Ethresh)/N_samples,histtype='step',linestyle='--',color="dodgerblue")
plt.hist(np.arccos(kinematics_scatter[1][:,-1])*1e3,bins=np.logspace(-4,4,100),weights=samples_scatter[2]*np.ones(N_samples)/N_samples*np.array(kinematics_scatter[0]>Ethresh),histtype='step',linestyle='--',color="orangered")
plt.plot([],[],color="black",ls="--",label="E > {} GeV".format(Ethresh))
plt.loglog()
plt.xlabel("Neutrino angle (mrad)")
plt.ylabel("Neutrinos per muon")
plt.legend()
plt.show()